**Variables, Domains, Constraints and Neighbors**

In [1]:
variables = ['India', 'Pakistan', 'Bangladesh', 'Nepal', 'Bhutan', 'SriLanka', 'Afghanistan']

domains = {v: ['Red', 'Green', 'Blue'] for v in variables}

constraints = [
    ('India',       'Pakistan'),
    ('India',       'Bangladesh'),
    ('India',       'Nepal'),
    ('India',       'Bhutan'),
    ('India',       'Afghanistan'),
    ('Pakistan',    'Afghanistan'),
    ('Nepal',       'Bhutan'),
]

neighbors = {v: set() for v in variables}
for (xi, xj) in constraints:
    neighbors[xi].add(xj)
    neighbors[xj].add(xi)

**REVISE Function and AC-3 (from Task 1)**

In [3]:
from collections import deque
import copy

def revise(domains, xi, xj):
    revised = False
    removed = []
    for x in domains[xi][:]:
        has_support = any(x != y for y in domains[xj])
        if not has_support:
            domains[xi].remove(x)
            removed.append(x)
            revised = True
    return revised, removed


def ac3(domains, constraints, neighbors, verbose=False):
    queue = deque()
    for (xi, xj) in constraints:
        queue.append((xi, xj))
        queue.append((xj, xi))
    while queue:
        (xi, xj) = queue.popleft()
        revised, _ = revise(domains, xi, xj)
        if revised:
            if len(domains[xi]) == 0:
                return False
            for xk in neighbors[xi]:
                if xk != xj:
                    queue.append((xk, xi))
    return True


# Run AC-3 and store reduced domains
reduced_domains = copy.deepcopy(domains)
ac3_result = ac3(reduced_domains, constraints, neighbors)

print('Reduced Domains after AC-3 (input to Backtracking):')
print()
print(f'{"Region":<15} {"Domain"}')
for v in variables:
    print(f'{v:<15} {reduced_domains[v]}')
print()
print(f'AC-3 Status: {"Success" if ac3_result else "Failure"}')

Reduced Domains after AC-3 (input to Backtracking):

Region          Domain
India           ['Red', 'Green', 'Blue']
Pakistan        ['Red', 'Green', 'Blue']
Bangladesh      ['Red', 'Green', 'Blue']
Nepal           ['Red', 'Green', 'Blue']
Bhutan          ['Red', 'Green', 'Blue']
SriLanka        ['Red', 'Green', 'Blue']
Afghanistan     ['Red', 'Green', 'Blue']

AC-3 Status: Success


**Implement the is_consistent Function**

In [4]:
def is_consistent(variable, value, assignment, neighbors):
    for neighbor in neighbors[variable]:
        if neighbor in assignment and assignment[neighbor] == value:
            return False
    return True


# Sanity Test
test_assignment = {'Pakistan': 'Red', 'Bangladesh': 'Blue'}

print('is_consistent — Sanity Test')
print('-' * 45)
r1 = is_consistent('India', 'Red',   test_assignment, neighbors)
r2 = is_consistent('India', 'Green', test_assignment, neighbors)
r3 = is_consistent('India', 'Blue',  test_assignment, neighbors)
print(f'Assign Red   to India (Pakistan=Red,  Bangladesh=Blue) → consistent: {r1}  (expected False)')
print(f'Assign Green to India (Pakistan=Red,  Bangladesh=Blue) → consistent: {r2}  (expected True)')
print(f'Assign Blue  to India (Pakistan=Red,  Bangladesh=Blue) → consistent: {r3}  (expected False)')
print()
print('is_consistent function works correctly.')

is_consistent — Sanity Test
---------------------------------------------
Assign Red   to India (Pakistan=Red,  Bangladesh=Blue) → consistent: False  (expected False)
Assign Green to India (Pakistan=Red,  Bangladesh=Blue) → consistent: True  (expected True)
Assign Blue  to India (Pakistan=Red,  Bangladesh=Blue) → consistent: False  (expected False)

is_consistent function works correctly.


**Implement the Backtracking Search**

In [5]:
counter = {'calls': 0, 'backtracks': 0}


def select_unassigned_variable(assignment, variables, domains):
    unassigned = [v for v in variables if v not in assignment]
    return min(unassigned, key=lambda var: len(domains[var]))


def backtrack(assignment, variables, domains, neighbors, counter):
    counter['calls'] += 1

    # Base case: all variables assigned
    if len(assignment) == len(variables):
        return assignment

    # Pick next variable using MRV heuristic
    var = select_unassigned_variable(assignment, variables, domains)

    for value in domains[var]:
        if is_consistent(var, value, assignment, neighbors):
            assignment[var] = value

            result = backtrack(assignment, variables, domains, neighbors, counter)
            if result is not None:
                return result

            del assignment[var]
            counter['backtracks'] += 1

    return None


print('Backtracking Search function defined.')
print('Variable selection: MRV heuristic (smallest domain first)')

Backtracking Search function defined.
Variable selection: MRV heuristic (smallest domain first)


**Run Backtracking Search on Reduced Domains**

In [6]:
counter = {'calls': 0, 'backtracks': 0}

bt_domains = copy.deepcopy(reduced_domains)

print('Running Backtracking Search on AC-3 reduced domains...')
print()

solution = backtrack({}, variables, bt_domains, neighbors, counter)

print(f'Recursive Calls  : {counter["calls"]}')
print(f'Backtracks Taken : {counter["backtracks"]}')
print()

if solution:
    print('Solution found!')
else:
    print('No solution exists for this CSP.')

Running Backtracking Search on AC-3 reduced domains...

Recursive Calls  : 8
Backtracks Taken : 0

Solution found!


**Validate Solution Against All Constraints**

In [7]:
def validate_solution(solution, constraints):
    violations = []
    for (xi, xj) in constraints:
        if solution.get(xi) == solution.get(xj):
            violations.append((xi, xj, solution.get(xi)))
    return len(violations) == 0, violations


if solution:
    all_valid, violations = validate_solution(solution, constraints)

    print('CONSTRAINT VALIDATION REPORT')
    print()
    print(f'{"Constraint":<30} {"Xi Color":<12} {"Xj Color":<12} {"Status"}')
    print('-' * 65)
    for (xi, xj) in constraints:
        ci, cj = solution[xi], solution[xj]
        status = 'OK' if ci != cj else 'VIOLATED'
        print(f'{xi + " != " + xj:<30} {ci:<12} {cj:<12} {status}')
    print('-' * 65)
    print()
    if all_valid:
        print('All constraints satisfied — solution is VALID!')
    else:
        print(f'{len(violations)} constraint(s) violated!')
else:
    print('No solution to validate.')

CONSTRAINT VALIDATION REPORT

Constraint                     Xi Color     Xj Color     Status
-----------------------------------------------------------------
India != Pakistan              Red          Green        OK
India != Bangladesh            Red          Green        OK
India != Nepal                 Red          Green        OK
India != Bhutan                Red          Blue         OK
India != Afghanistan           Red          Blue         OK
Pakistan != Afghanistan        Green        Blue         OK
Nepal != Bhutan                Green        Blue         OK
-----------------------------------------------------------------

All constraints satisfied — solution is VALID!


**Final Valid Coloring of the Map**

In [8]:
FULL_NAMES = {
    'India':       'India              ',
    'Pakistan':    'Pakistan           ',
    'Bangladesh':  'Bangladesh         ',
    'Nepal':       'Nepal              ',
    'Bhutan':      'Bhutan             ',
    'SriLanka':    'Sri Lanka          ',
    'Afghanistan': 'Afghanistan        ',
}

if solution:
    print('FINAL VALID MAP COLORING — South Asia')
    print()
    print(f'{"Region":<15} {"Full Name":<22} {"Assigned Color"}')
    print('-' * 50)
    for var in variables:
        color     = solution[var]
        full_name = FULL_NAMES[var]
        print(f'{var:<15} {full_name:<22} {color}')
    print('-' * 50)
    print()
    print('Note: SriLanka is an island — any color is valid (no neighbors).')
else:
    print('No solution was found.')

FINAL VALID MAP COLORING — South Asia

Region          Full Name              Assigned Color
--------------------------------------------------
India           India                  Red
Pakistan        Pakistan               Green
Bangladesh      Bangladesh             Green
Nepal           Nepal                  Green
Bhutan          Bhutan                 Blue
SriLanka        Sri Lanka              Red
Afghanistan     Afghanistan            Blue
--------------------------------------------------

Note: SriLanka is an island — any color is valid (no neighbors).
